In [1]:
# ============================================================
# CELL 1 — CONFIG + IMPORTS + SEED
# ============================================================

# -------- CONFIG --------
MODE = "A"          # "A", "B", "C", "D"
EPOCHS = 400
SEED = 4           # use 1, 2, 3 for multiple runs

CFG = {
    "batch_size": 256,
    "lr": 3e-4,
    "temperature": 0.5,
    "proj_dim": 128,
    "lambda_comp": 0.5,
    "img_size": 96,
    "blur_kernel": 23,
    "sigma": 2.0,
    "data_dir": "/kaggle/working/data",
    "save_dir": "/kaggle/working/checkpoints",
}

# -------- IMPORTS --------
import os, time, datetime
import random, numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as T
from torchvision.models import resnet18
from torchvision.datasets import STL10
from torch.cuda.amp import GradScaler
from torch.cuda.amp import autocast, GradScaler

# -------- DEVICE --------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------- SEED --------
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"MODE: {MODE} | SEED: {SEED} | DEVICE: {device}")

# -------- DIRS --------
os.makedirs(CFG["data_dir"], exist_ok=True)
os.makedirs(CFG["save_dir"], exist_ok=True)

MODE: A | SEED: 4 | DEVICE: cuda


In [2]:
unlabeled = STL10(CFG["data_dir"], split="unlabeled", download=True)
print("Unlabeled size:", len(unlabeled))

100%|██████████| 2.64G/2.64G [01:34<00:00, 28.0MB/s] 


Unlabeled size: 100000


In [3]:
class Residual(nn.Module):
    def __init__(self):
        super().__init__()
        k = CFG["blur_kernel"]
        s = CFG["sigma"]

        coords = torch.arange(k) - k//2
        g = torch.exp(-(coords**2)/(2*s**2))
        g /= g.sum()
        kernel = (g[:,None]*g[None,:]).unsqueeze(0).unsqueeze(0)

        self.register_buffer("kernel", kernel)
        self.k = k

    def forward(self,x):
        C = x.shape[1]
        k = self.kernel.expand(C,1,self.k,self.k)
        blur = F.conv2d(x,k,padding=self.k//2,groups=C)

        r = x - blur
        mean = r.mean([1,2,3],keepdim=True)
        std  = r.std([1,2,3],keepdim=True) + 1e-6

        return (r - mean)/std

res_op = Residual().to(device)

In [4]:
augment = T.Compose([
    T.RandomResizedCrop(CFG["img_size"], scale=(0.2,1.0)),
    T.RandomHorizontalFlip(),
    T.ToTensor()
])

normalize = T.Normalize([0.485,0.456,0.406],
                        [0.229,0.224,0.225])

In [5]:
class SSLDataset(Dataset):
    def __init__(self,data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self,idx):
        img,_ = self.data[idx]

        v1 = augment(img)
        v2 = augment(img)

        if MODE == "A":
            return normalize(v1), normalize(v2)

        elif MODE == "B":
            return v1, v2   # no residual here

        else:  # C, D
            return normalize(v1), normalize(v2), v1, v2

In [6]:
loader = DataLoader(
    SSLDataset(unlabeled),
    batch_size=CFG["batch_size"],
    shuffle=True,
    num_workers=2,
    drop_last=True
)

print("Batches per epoch:", len(loader))

Batches per epoch: 390


In [7]:
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = nn.Sequential(
            *list(resnet18(weights=None).children())[:-1]
        )

    def forward(self,x):
        return self.backbone(x).flatten(1)


class Projector(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(512,512),
            nn.ReLU(),
            nn.Linear(512,CFG["proj_dim"])
        )

    def forward(self,x):
        return self.net(x)


class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = Encoder()
        self.proj_o = Projector()
        self.proj_r = Projector()

    def forward(self,x,r):
        f_o = self.enc(x)
        f_r = self.enc(r)
        return self.proj_o(f_o), self.proj_r(f_r), f_o, f_r


model = Model().to(device)

In [8]:
def ntx(z1,z2):
    B = z1.size(0)

    z = torch.cat([z1,z2],0)
    z = F.normalize(z,dim=1)

    sim = z @ z.T / CFG["temperature"]

    mask = torch.eye(2*B,device=z.device).bool()
    sim.masked_fill_(mask,-1e4)

    target = torch.cat([
        torch.arange(B,2*B),
        torch.arange(0,B)
    ]).to(z.device)

    return F.cross_entropy(sim,target)

In [9]:
optimizer = torch.optim.Adam(model.parameters(), lr=CFG["lr"])
scaler = GradScaler()

ckpt_path = f"{CFG['save_dir']}/{MODE}.pt"
start_epoch = 1

if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["opt"])
    start_epoch = ckpt["epoch"] + 1
    print(f"Resumed from epoch {start_epoch}")

/tmp/ipykernel_55/596967519.py:2: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [10]:
start_time = time.time()

for epoch in range(start_epoch, EPOCHS+1):

    model.train()
    total_loss = 0
    ep_start = time.time()

    for batch in loader:
        optimizer.zero_grad()

        with torch.amp.autocast("cuda"):

            # ======================
            # MODE A (baseline)
            # ======================
            if MODE == "A":
                v1, v2 = [b.to(device) for b in batch]

                z1 = model.proj_o(model.enc(v1))
                z2 = model.proj_o(model.enc(v2))

                loss = ntx(z1, z2)

            # ======================
            # MODE B (residual only)
            # ======================
            elif MODE == "B":
                v1, v2 = [b.to(device) for b in batch]

                r1 = res_op(v1)
                r2 = res_op(v2)

                z1 = model.proj_r(model.enc(r1))
                z2 = model.proj_r(model.enc(r2))

                loss = ntx(z1, z2)

            # ======================
            # MODE C / D (dual)
            # ======================
            else:
                v1, v2, v1_raw, v2_raw = [b.to(device) for b in batch]

                # residual computed ON GPU
                r1 = res_op(v1_raw)
                r2 = res_op(v2_raw)

                z_o1, z_r1, f_o1, f_r1 = model(v1, r1)
                z_o2, z_r2, f_o2, f_r2 = model(v2, r2)

                loss = ntx(z_o1, z_o2) + ntx(z_r1, z_r2)

                # ===== MODE D (novel) =====
                if MODE == "D":
                    f_o1 = F.normalize(f_o1, dim=1)
                    f_o2 = F.normalize(f_o2, dim=1)
                    f_r1 = F.normalize(f_r1, dim=1)
                    f_r2 = F.normalize(f_r2, dim=1)

                    cos = ((f_o1 * f_r1).sum(1) +
                           (f_o2 * f_r2).sum(1)).mean() / 2

                    loss -= CFG["lambda_comp"] * cos

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    # ===== timing =====
    ep_time = time.time() - ep_start
    remaining = ep_time * (EPOCHS - epoch)
    eta = datetime.datetime.now() + datetime.timedelta(seconds=remaining)

    print(f"[{MODE}] Epoch {epoch}/{EPOCHS} | "
          f"loss {total_loss/len(loader):.4f} | "
          f"{ep_time:.1f}s | ETA {eta.strftime('%H:%M:%S')}")

    # ===== save checkpoint =====
    torch.save({
        "epoch": epoch,
        "model": model.state_dict(),
        "opt": optimizer.state_dict()
    }, ckpt_path)

[A] Epoch 1/400 | loss 4.6386 | 98.6s | ETA 00:38:57
[A] Epoch 2/400 | loss 4.5182 | 98.2s | ETA 00:36:00
[A] Epoch 3/400 | loss 4.4882 | 98.6s | ETA 00:38:42
[A] Epoch 4/400 | loss 4.4707 | 96.1s | ETA 00:22:24
[A] Epoch 5/400 | loss 4.4602 | 95.9s | ETA 00:21:12
[A] Epoch 6/400 | loss 4.4505 | 98.6s | ETA 00:39:01
[A] Epoch 7/400 | loss 4.4438 | 95.5s | ETA 00:18:15
[A] Epoch 8/400 | loss 4.4390 | 96.9s | ETA 00:27:43
[A] Epoch 9/400 | loss 4.4334 | 95.6s | ETA 00:19:23
[A] Epoch 10/400 | loss 4.4287 | 94.9s | ETA 00:14:35
[A] Epoch 11/400 | loss 4.4258 | 97.9s | ETA 00:34:15
[A] Epoch 12/400 | loss 4.4226 | 95.7s | ETA 00:20:02
[A] Epoch 13/400 | loss 4.4185 | 95.0s | ETA 00:15:33
[A] Epoch 14/400 | loss 4.4164 | 98.2s | ETA 00:36:06
[A] Epoch 15/400 | loss 4.4150 | 96.8s | ETA 00:26:39
[A] Epoch 16/400 | loss 4.4113 | 97.0s | ETA 00:28:00
[A] Epoch 17/400 | loss 4.4101 | 96.4s | ETA 00:24:21
[A] Epoch 18/400 | loss 4.4073 | 94.9s | ETA 00:15:03
[A] Epoch 19/400 | loss 4.4062 | 97.5

In [11]:
end_time = time.time()
total_time = end_time - start_time

hours = int(total_time // 3600)
minutes = int((total_time % 3600) // 60)
seconds = int(total_time % 60)

print("\n===== TRAINING COMPLETE =====")
print(f"MODE: {MODE}")
print(f"Final Epoch: {epoch}")
print(f"Final Loss: {total_loss/len(loader):.4f}")
print(f"Total Training Time: {hours}h {minutes}m {seconds}s")
print(f"Checkpoint saved at: {ckpt_path}")


===== TRAINING COMPLETE =====
MODE: A
Final Epoch: 400
Final Loss: 4.3208
Total Training Time: 10h 46m 18s
Checkpoint saved at: /kaggle/working/checkpoints/A.pt
